# w9_i2ce_continue.ipynb — i2ce@4096 五折续训 2000 -> 4000ep

User: 不要猜想，要事实。The five final_experiment i2ce@4096 CV towers
continue from their ep2000 checkpoints to 4000 (cv-worker EXTEND fallback:
weights from the newest ckpt, fresh opt/rng, seed=fold semantics unchanged).
Only towers that HAVE reached ep2000 are extended -- unfinished folds print
[wait] and are skipped (re-run this notebook after final_experiment's 4096
column completes; claims coordinate if both run at once). zsbest re-picks
over the FULL 50..4000 curve by cvsel. ~20h/tower on A100-80G, solo per GPU
(grad gallery ~45G); 5 towers = one wave on 6 GPUs. The readout prints
facts only: old-best (<=2000) vs new-best (<=4000) per fold + tail windows.
RECIPES is a constant -- add "wcle_ce_cetf" there if the paired control
should follow.


In [ ]:
# constants
import os

REPO = "/workspace/stable-query-latent"
URL = "https://github.com/Nice9Tian/stable-query-latent.git"
DATA_SRC = "/workspace/fusion_cache_w9"
DATA_RAM = "/dev/shm/fusion_cache_w9"
OUT_DIR = "/workspace/w9_cv_out"       # CV campaign out dir (same as final)

RECIPES = ["wcle_i2ce_icetf"]          # add "wcle_ce_cetf" for the paired control
CAPS = [4096]
N_FOLDS = 5
EPOCHS_BASE = 2000                     # only towers at this budget get extended
EPOCHS = 4000
CKPT_EVERY, CKPT_SEEDS, TOPUP_SEEDS = 50, 2, 10   # ZS-only: seeds unused

SAFETY = 0.85
RESERVE_GIB = 1.5
MAX_CAP_48G = 2048                     # <60G pods have nothing to do here

def nm_of(r, k, cap):
    return f"w9cv_{r}_fold{k}" + (f"_g{cap}" if cap != 512 else "")

os.makedirs(OUT_DIR, exist_ok=True)
print("candidates:", len(RECIPES) * N_FOLDS * len(CAPS), f"towers -> {EPOCHS}ep")


In [ ]:
# FORCE-sync repo to origin/main.
import os, importlib.util
if not os.path.isdir(os.path.join(REPO, ".git")):
    !git clone {URL} {REPO}
%cd {REPO}
!git remote set-url origin {URL}
!git fetch origin main
!git reset --hard origin/main
!git rev-parse --short HEAD
for pkg in ("sklearn", "scipy"):
    if importlib.util.find_spec(pkg) is None:
        !pip -q install scikit-learn scipy
        break
import sys
if REPO not in sys.path:
    sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, "Pod"))
import w9_jobs as J
print("machinery loaded")


In [ ]:
# Stage the corpus into RAM (same file set as w9_a100.ipynb).
import shutil
from pathlib import Path
REQUIRED = ["games.npz", "wiki_eval.npz", "wscan_gal_rev.npz",
            "wscan_pool_rev.npy", "wscan_pool_rev_rid.npy", "wscan_pool_rev_len.npy",
            "ss_queries_rev.npz", "ss_queries_rev_S.npy",
            "wiki_clean_views.npz", "sp_raw_views.npz", "tag_labels.npz",
            "wiki_eval_split.json", "_tag_splitM.json"]
src = Path(DATA_SRC)
missing = [f for f in REQUIRED if not (src / f).exists()]
assert not missing, f"missing in {DATA_SRC}: {missing}"
dst = Path(DATA_RAM)
dst.mkdir(parents=True, exist_ok=True)
for f in REQUIRED:
    s, d = src / f, dst / f
    if not d.exists() or d.stat().st_size != s.stat().st_size:
        print(f"staging {f} ({s.stat().st_size/1e9:.2f} GB) ...", flush=True)
        shutil.copyfile(s, d)
DATA_DIR = str(dst)
print("corpus in RAM:", DATA_DIR)

In [ ]:
# Full pool: must be READY on the volume; stage onto fast local storage.
import os, time
from pathlib import Path
from Pod.h5_staging import parallel_copy

ready = Path(DATA_SRC) / "full_pool_READY"
assert ready.exists(), "full pool not READY -- run a campaign notebook's build cell once"
src_v = Path(DATA_SRC) / "full_pool_fp16.npy"
src_m = Path(DATA_SRC) / "full_pool_meta.npz"
need = src_v.stat().st_size + (5 << 30)

def _free(p):
    st = os.statvfs(p)
    return st.f_bavail * st.f_frsize

dest_dir = None
for cand in ("/dev/shm", "/root/data", "/root"):
    Path(cand).mkdir(parents=True, exist_ok=True)
    if _free(cand) > need:
        dest_dir = Path(cand)
        break
if dest_dir is None:
    print("WARNING: no local space -- workers will mmap the NETWORK VOLUME copy.")
    FULL_POOL_PATH = str(src_v)
else:
    dst_v = dest_dir / "full_pool_fp16.npy"
    if dst_v.exists() and dst_v.stat().st_size == src_v.stat().st_size:
        print("local full pool already staged:", dst_v)
    else:
        t0 = time.time()
        tmp = dst_v.with_name(dst_v.name + ".copying")
        print(f"staging {src_v.stat().st_size/2**30:.0f} GiB -> {dst_v} ...", flush=True)
        parallel_copy(src_v, tmp, workers=8)
        os.replace(tmp, dst_v)
        print(f"staged in {(time.time()-t0)/60:.1f} min", flush=True)
    import shutil
    shutil.copyfile(src_m, dest_dir / "full_pool_meta.npz")
    FULL_POOL_PATH = str(dst_v)
print("FULL_POOL_PATH =", FULL_POOL_PATH)


In [ ]:
# Extend drain: only ep2000-complete towers; VRAM scheduler, solo @4096.
import os, subprocess, tempfile, threading, time
from pathlib import Path

cdir = Path(OUT_DIR) / "claims"
logd = Path(OUT_DIR) / "logs"
logd.mkdir(parents=True, exist_ok=True)
# measure runs write to a LOCAL scratch dir (never the shared volume)
MEAS_OUT = os.path.join(tempfile.gettempdir(), "w9_measure_out")
os.makedirs(MEAS_OUT, exist_ok=True)
gpus = J.detect_gpus()

def _smi_mib(field, g):
    out = subprocess.check_output(
        ["nvidia-smi", f"--query-gpu={field}", "--format=csv,noheader,nounits",
         "-i", str(g)]).decode().strip().split("\n")[0]
    return int(out) * 2**20

free = {g: _smi_mib("memory.free", g) for g in gpus}
budget = {g: int(free[g] * SAFETY - RESERVE_GIB * 2**30) for g in gpus}
vram_gib = min(free.values()) / 2**30
CAP_CEIL = MAX_CAP_48G if vram_gib < 60 else 10**9
print(f"[vram] budgets {[f'{budget[g] / 2**30:.0f}G' for g in gpus]}  ceiling "
      f"{CAP_CEIL if CAP_CEIL < 10**9 else 'none'}")

todo0 = []
for cap in CAPS:
    for r in RECIPES:
        for k in range(N_FOLDS):
            nm = nm_of(r, k, cap)
            if cap > CAP_CEIL:
                print(f"[skip-vram] {nm}"); continue
            if (Path(OUT_DIR) / f"tower_{nm}_fp_ep{EPOCHS}.npz").exists():
                print(f"[skip] {nm} already at {EPOCHS}"); continue
            if not (Path(OUT_DIR) / f"tower_{nm}_fp_ep{EPOCHS_BASE}.npz").exists():
                print(f"[wait] {nm} not yet at {EPOCHS_BASE} -- run "
                      f"final_experiment first / re-run later"); continue
            todo0.append((r, k, cap, nm))
print(f"{len(todo0)} towers to extend")

cost = {}
for cap in sorted({c for _r, _k, c, _n in todo0}):
    tf = Path(tempfile.gettempdir()) / f"w9cvx_vram_{cap}.txt"
    tf.unlink(missing_ok=True)
    cmd = ["python", "-u", J.CV_WORKER, "--data-dir", DATA_DIR, "--out-dir",
           MEAS_OUT, "--repo", REPO, "--arm", RECIPES[0], "--fold", "0",
           "--n-folds", str(N_FOLDS), "--anchor-cap", str(cap),
           "--epochs", "1", "--full-pool", "--full-pool-path", FULL_POOL_PATH,
           "--measure-vram", str(tf)]
    print(f"[warmup] cap {cap} ...", flush=True)
    with open(logd / f"measure_cvx_g{cap}.log", "w") as fh:
        subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                       env=dict(os.environ, CUDA_VISIBLE_DEVICES=gpus[0]))
    cost[cap] = int(tf.read_text()) if tf.exists() else budget[gpus[0]] + 1
    print(f"[warmup] cap {cap}: {cost[cap] / 2**30:.2f}G", flush=True)

todo = sorted(((r, k, cap, nm, cost[cap]) for r, k, cap, nm in todo0),
              key=lambda x: -x[4])
now_used = {g: 0 for g in gpus}
fails = []
cvn = threading.Condition()

def run_job(g, r, k, cap, nm, c):
    try:
        if not J.try_claim(cdir, nm):
            print(f"[claim] {nm} held elsewhere -- skipped", flush=True); return
        cmd = ["python", "-u", J.CV_WORKER, "--data-dir", DATA_DIR, "--out-dir",
               OUT_DIR, "--repo", REPO, "--arm", r, "--fold", str(k),
               "--n-folds", str(N_FOLDS), "--anchor-cap", str(cap),
               "--epochs", str(EPOCHS), "--ckpt-every", str(CKPT_EVERY),
               "--ckpt-seeds", str(CKPT_SEEDS), "--topup-seeds", str(TOPUP_SEEDS),
               "--full-pool", "--full-pool-path", FULL_POOL_PATH,
               "--claim-file", str(cdir / f"{nm}.claim")]
        t0 = time.time()
        with open(logd / f"{r}_fold{k}_g{cap}_ext{EPOCHS}.log", "w") as fh:
            p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT,
                               env=dict(os.environ, CUDA_VISIBLE_DEVICES=g))
        if p.returncode != 0:
            (cdir / f"{nm}.claim").unlink(missing_ok=True); fails.append(nm)
        print(f"[gpu{g}] {'ok' if p.returncode == 0 else 'FAIL'} {nm} "
              f"[{(time.time() - t0) / 3600:.1f}h]", flush=True)
    finally:
        with cvn:
            now_used[g] -= c
            cvn.notify_all()

stop_evt = threading.Event()
threading.Thread(target=J._monitor, args=([logd], stop_evt), daemon=True).start()
active = []
with cvn:
    pending = list(todo)
    while pending or any(t.is_alive() for t in active):
        prog = False
        i = 0
        while i < len(pending):
            r, k, cap, nm, c = pending[i]
            fit = [g for g in gpus if now_used[g] + c <= budget[g] or now_used[g] == 0]
            if not fit:
                i += 1; continue
            g = min(fit, key=lambda g: now_used[g])
            now_used[g] += c
            th = threading.Thread(target=run_job, args=(g, r, k, cap, nm, c),
                                  daemon=True)
            active.append(th); th.start(); pending.pop(i)
            print(f"[sched] {nm} -> gpu{g} ({c / 2**30:.1f}G)", flush=True)
            prog = True
        active = [t for t in active if t.is_alive()]
        if not prog:
            cvn.wait(timeout=3)
stop_evt.set()
for t in active:
    t.join()
print(f"extension drained; {len(fails)} failed")
for nm in fails:
    print("  FAILED:", nm)


In [ ]:
# FACTS: per fold, best<=2000 vs best<=4000 (cvsel-selected from the traj)
# and tail windows [1500-2000] vs [3500-4000]. No interpretation.
import json
import numpy as np
from pathlib import Path

def wmean(tr, eps, key, lo, hi):
    v = [tr[e][key] for e in eps if lo <= int(e[2:]) <= hi and key in tr[e]]
    return float(np.mean(v)) if v else float("nan")

for r in RECIPES:
    for cap in CAPS:
        print(f"===== {r.replace('wcle_', '').replace('_icetf', '').replace('_cetf', '')}@{cap} =====")
        rows = []
        for k in range(N_FOLDS):
            nm = nm_of(r, k, cap)
            p = Path(OUT_DIR) / f"zs_traj_{nm}_fp.json"
            if not p.exists():
                print(f"  fold{k}: (no traj)"); continue
            tr = json.loads(p.read_text())
            eps = sorted(tr, key=lambda e: int(e[2:]))
            cand = [e for e in eps if "cvsel" in tr[e]]
            def best(upto):
                cs = [e for e in cand if int(e[2:]) <= upto]
                if not cs:
                    return None
                b = max(cs, key=lambda e: (tr[e]["cvsel"], -int(e[2:])))
                return b
            b1, b2 = best(2000), best(10**9)
            if not b1 or not b2:
                print(f"  fold{k}: (no cvsel keys)"); continue
            d1, d2 = tr[b1], tr[b2]
            rows.append((d1, d2))
            print(f"  fold{k}: best<=2000 ep{b1[2:]:>4} non {d1['nm_noname']:.3f} "
                  f"tag {d1['tag_noname']:.3f} cvsel {d1['cvsel']:.3f}   ||   "
                  f"best<=4000 ep{b2[2:]:>4} non {d2['nm_noname']:.3f} "
                  f"tag {d2['tag_noname']:.3f} cvsel {d2['cvsel']:.3f}")
            for key in ("nm_noname", "tag_noname", "cvsel"):
                a = wmean(tr, eps, key, 1500, 2000)
                b = wmean(tr, eps, key, 3500, 4000)
                print(f"          {key:10s} win[1500-2000] {a:.3f} -> win[3500-4000] {b:.3f} ({b - a:+.3f})")
        if rows:
            for key, lab in (("nm_noname", "non@1"), ("tag_noname", "tag_non")):
                d_old = np.mean([a[key] for a, _b in rows])
                d_new = np.mean([b[key] for _a, b in rows])
                print(f"  MEAN over {len(rows)} folds: {lab} best<=2000 {d_old:.3f} "
                      f"-> best<=4000 {d_new:.3f} ({d_new - d_old:+.3f})")


In [ ]:
# AUTO-STOP: stop THIS pod when the queue has finished (results live on the
# network volume; idle GPU time is pure waste). Uses the hardened ladder in
# VICReg_review/pod_selfstop.py. Set AUTO_STOP=False to keep the pod alive.
AUTO_STOP = True
if AUTO_STOP:
    import sys
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from VICReg_review import pod_selfstop
    if fails:
        print(f"NOTE: {len(fails)} job(s) FAILED -- logs in {OUT_DIR}/logs; "
              "stopping anyway to avoid idle burn.")
    pod_id, api_key, ctl = pod_selfstop.preflight("")
    pod_selfstop.stop_pod(pod_id, api_key, ctl)
else:
    print("AUTO_STOP disabled -- remember to stop the pod yourself.")